# 장면 모델 학습 (crosswalk, road)

`training/aihub_scene_extract.py`로 만든 `aihub_scene_v1.zip`으로 YOLOv8-seg 2클래스 모델을 학습함.

- 런타임 → 런타임 유형 변경 → **GPU(T4)** 선택 후 위에서부터 실행함
- 결과(가중치)는 Google Drive `tactilecheck/runs/scene_v1/`에 저장되므로 연결이 끊겨도 남음
- 끊겼으면 1~3번 셀을 다시 실행하고 **5번(이어서 학습)** 을 실행함

In [ ]:
# 1. GPU 확인
!nvidia-smi

In [ ]:
# 2. Drive 연결, 데이터를 Colab 로컬 디스크로 복사 후 압축 해제 (Drive에서 직접 읽으면 느림)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/tactilecheck'
!mkdir -p {DRIVE}/runs
!cp {DRIVE}/aihub_scene_v1.zip /content/ && cd /content && unzip -q -o aihub_scene_v1.zip
!ls {DRIVE}  # 여기에 aihub_scene_v1.zip이 보여야 함
# data.yaml의 경로를 Colab 절대경로로 다시 씀 (ultralytics는 상대경로를 /content/datasets 기준으로 찾음)
open('/content/aihub_scene_v1/data.yaml', 'w').write(
    'path: /content/aihub_scene_v1\ntrain: images/train\nval: images/val\nnames:\n  0: crosswalk\n  1: road\n')
!ls /content/aihub_scene_v1/images/train | wc -l; ls /content/aihub_scene_v1/images/val | wc -l  # 5558, 1103

In [ ]:
# 3. 설치
!pip -q install ultralytics

In [ ]:
# 4. 학습 (T4 기준 에폭당 약 2~3분)
from ultralytics import YOLO
model = YOLO('yolov8s-seg.pt')
model.train(
    data='/content/aihub_scene_v1/data.yaml',
    imgsz=640, epochs=40, batch=16, patience=10,
    project=f'{DRIVE}/runs', name='scene_v1', exist_ok=True,
    save_period=5, workers=2,
)

In [ ]:
# 5. (연결이 끊겼을 때만) 이어서 학습
# from ultralytics import YOLO
# YOLO(f'{DRIVE}/runs/scene_v1/weights/last.pt').train(resume=True)

In [ ]:
# 6. 검증: 클래스별 mask mAP
from ultralytics import YOLO
best = YOLO(f'{DRIVE}/runs/scene_v1/weights/best.pt')
m = best.val(data='/content/aihub_scene_v1/data.yaml', imgsz=640, split='val')
for i, n in best.names.items():
    print(f'{n:10s} mask mAP50 {m.seg.ap50[i]:.3f}  mAP50-95 {m.seg.ap[i]:.3f}')

In [ ]:
# 7. val 사진 몇 장에 예측을 그려 봄
import glob
from IPython.display import Image, display
imgs = sorted(glob.glob('/content/aihub_scene_v1/images/val/*.jpg'))[::120][:8]
for r in best.predict(imgs, imgsz=640, conf=0.3, save=True, project='/content/pred', name='val', exist_ok=True):
    pass
for p in sorted(glob.glob('/content/pred/val/*.jpg')):
    display(Image(p, width=640))